In [0]:
from pyspark.sql import functions as F

# Step 1: load the raw data
bronze = spark.table("carbon_intensity.bronze.intensity_raw")

# Step 2: tell Spark the shape of the JSON so it can read it
schema = "struct<data:array<struct<from:string,to:string,intensity:struct<forecast:int,actual:int,index:string>>>>"
bronze = bronze.withColumn("parsed", F.from_json("raw_json", schema))

# Step 3: each API pull has multiple time periods inside it - split them into separate rows
bronze = bronze.select(F.explode("parsed.data").alias("d"), "ingested_at")

# Step 4: pull out the fields we actually want, with simple names
silver = bronze.select(
    F.to_timestamp("d.from", "yyyy-MM-dd'T'HH:mm'Z'").alias("period_start"),
    F.col("d.intensity.forecast").alias("forecast"),
    F.col("d.intensity.actual").alias("actual"),
    F.col("d.intensity.index").alias("intensity_band")
)
silver.show(5)

In [0]:
# Step 5: use actual if we have it, otherwise use forecast
silver = silver.withColumn("intensity_value", F.coalesce("actual", "forecast"))

silver.show(5)

In [0]:
# Step 6: pull out the hour and day, since that's what we need to answer "greenest time"
silver = silver.withColumn("hour_of_day", F.hour("period_start"))
silver = silver.withColumn("day_of_week", F.date_format("period_start", "E"))

silver.show(5)

In [0]:
# Step 7: remove exact duplicate rows (simple version - not perfect, but easy to read)
silver = silver.dropDuplicates(["period_start"])
silver.show(5)


In [0]:

# Step 8: save it
silver.write.mode("overwrite").saveAsTable("carbon_intensity.silver.intensity_clean")
silver.show(5)

In [0]:
# Step 9: check it looks right
print("Row count:", silver.count())
